In [ ]:
# ======================================================
# Kushinada: 特徴量CSV → 感情（分類）・強度（回帰）推定     (CSJコーパスを入力)
# ＋ ラベル変換付き完全版
# ======================================================

import os
import pandas as pd
import numpy as np
import joblib

# ================================
# パス設定
# ================================
FEATURE_CSV = "/home/mitani/s3prl/s3prl/csj_kushinada_features.csv"

MODEL_DIR = "/home/mitani/results_kushinada_emo_int"

# OGVCの感情・強度で学習済みのモデルをロード
EMO_MODEL_PATH = os.path.join(MODEL_DIR, "kusinada_csj_emo_model-s3prl-jtes_out0.pkl")
INT_MODEL_PATH = os.path.join(MODEL_DIR, "kusinada_csj_int_model-s3prl-jtes_out0.pkl")
FEATURE_COLS_PATH = os.path.join(MODEL_DIR, "kusinada_feature_columns_out0.pkl")

OUTPUT_CSV = "pred_csj-kushinada_EmoLabel_out0.csv"

# ================================
# ラベル定義
# ================================
EMOTION_LABELS = {
    "JOY": 0, "ACC": 1, "FEA": 2, "SUR": 3,
    "SAD": 4, "DIS": 5, "ANG": 6, "ANT": 7,
    "NEU": 8, "OTH": 9
}

# 数値 → ラベル
ID2EMO = {v: k for k, v in EMOTION_LABELS.items()}

INTENSITY_MAX = 3

# ================================
# Step1: モデルロード
# ================================
emo_model = joblib.load(EMO_MODEL_PATH)
int_model = joblib.load(INT_MODEL_PATH)
feature_cols = joblib.load(FEATURE_COLS_PATH)

print("=== モデル読み込み完了 ===")

# ================================
# Step2: 特徴量読み込み
# ================================
df = pd.read_csv(FEATURE_CSV)

# filename以外を特徴量に
X = df.drop(columns=["filename"])

# 列順を揃える（超重要）
X = X[feature_cols]

# ================================
# Step3: 推論
# ================================
emo_pred = emo_model.predict(X)

# 回帰 → 元スケールへ戻す
int_pred = int_model.predict(X) * INTENSITY_MAX

# 丸め（0〜3）
int_pred_round = np.clip(np.round(int_pred), 0, INTENSITY_MAX).astype(int)

# ================================
# Step4: ラベル変換
# ================================
emo_pred_label = pd.Series(emo_pred).map(ID2EMO)

# ================================
# Step5: 保存
# ================================
result_df = pd.DataFrame({
    "filename": df["filename"],
    "emotion_pred": emo_pred,
    "emotion_label": emo_pred_label,
    "intensity_pred": int_pred,
    "intensity_pred_round": int_pred_round
})

result_df.to_csv(OUTPUT_CSV, index=False)

print("=== 推論完了 ===")
print(result_df.head())

=== モデル読み込み完了 ===
=== 推論完了 ===
     filename  emotion_pred emotion_label  intensity_pred  \
0  0001_L.wav             1           ACC        3.854964   
1  0002_L.wav             2           FEA        0.007986   
2  0003_R.wav             1           ACC        2.591175   
3  0004_L.wav             0           JOY        2.292682   
4  0005_R.wav             4           SAD        0.552191   

   intensity_pred_round  
0                     3  
1                     0  
2                     3  
3                     2  
4                     1  
